# Milestone 3 — Baseline Reproduction
### Team SixtyFive.AI · CSCI 3052U Machine Learning I
**Project:** Out-of-Distribution Generalization in Satellite Land Cover Classification: A Geographic Domain-Shift Approach

**Purpose.** Reproduce the team's majority-class and RGB PCA + Random Forest baselines on EuroSAT. Compare random stratified development validation against a candidate geographic out-of-distribution (OOD) test set.

**Implementation source:** `milestone3-backup/src/baseline.py` and `src/run_rgb_baseline.py` (team repository). Baseline settings and procedures are preserved in the cells below. This notebook has **not** been run on the full dataset in this deliverable: the provided ZIP contains no RGB image dataset or rebuilt geographic manifest. Results shown after running the training cells will be newly computed, not invented.

**Run sequence:** Install dependencies → place the rebuilt manifest and RGB images as described → **Kernel > Restart & Run All**. No model should be fitted to OOD test labels.

## 1. Environment and reproducibility

Required packages: `numpy`, `pandas`, `scikit-learn`, `Pillow`, `matplotlib`, `jupyter` (for the notebook interface). Recommended: Python 3.10–3.12. Run from the project repository; the notebook locates the repository root automatically. Seed = **42**.

Install once from the project root (terminal):
```bash
python -m pip install numpy pandas scikit-learn pillow matplotlib jupyter
```

The **rebuilt geographic manifest** is generated by the team's `src/build_geographic_manifest.py` on `milestone3-backup`, which requires `outputs/geotiff_metadata.csv`, Natural Earth boundary files and `pyshp`, `shapely`, `pyproj`. This notebook consumes that manifest rather than replacing its geographic methodology.

In [ ]:
from pathlib import Path
import sys
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.dummy import DummyClassifier
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 42
IMAGE_SIZE = (16, 16)
PCA_COMPONENTS = 100
N_TREES = 200

# Works when executed from repo root or notebooks/.
CWD = Path.cwd().resolve()
if (CWD / 'outputs').is_dir():
    ROOT = CWD
elif (CWD.parent / 'outputs').is_dir():
    ROOT = CWD.parent
else:
    raise FileNotFoundError('Run inside the team project folder (or its notebooks/ directory).')

OUTPUTS = ROOT / 'outputs'
MANIFEST_PATH = OUTPUTS / 'geographic_manifest_rebuilt.csv'
print('Project root:', ROOT)
print('Python:', platform.python_version(), '| scikit-learn:', __import__('sklearn').__version__)
print('Seed:', SEED)

## 2. Load and audit the geographic manifest

The manifest assigns candidate samples to `development`, `ood_test`, `excluded_region`, or `unresolved`. In the team's approach, Western/Central Europe is used for development and Southern/Eastern Europe is the candidate OOD test. Border/coast ambiguity and unassigned regions are excluded from training/evaluation.

If the manifest is missing, first check out the team's `milestone3-backup` branch or copy **its generated CSV** into `outputs/`. Rebuilding the manifest requires the original geographic metadata and Natural Earth boundaries; do not silently substitute the older `preliminary_region_manifest.csv`.

In [ ]:
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(
        f'Missing {MANIFEST_PATH}. Copy the validated geographic_manifest_rebuilt.csv '
        'from milestone3-backup/outputs or run the team manifest builder first.'
    )
manifest = pd.read_csv(MANIFEST_PATH)
needed = {'sample_id', 'class', 'candidate_partition'}
assert needed.issubset(manifest.columns), f'Missing manifest columns: {needed - set(manifest.columns)}'
assert manifest['sample_id'].notna().all() and manifest['sample_id'].is_unique, 'Sample IDs must be unique.'
print('All manifest rows:', len(manifest))
print('\nPartition counts:')
print(manifest['candidate_partition'].value_counts(dropna=False).to_string())
print('\nClass counts by partition:')
display(pd.crosstab(manifest['class'], manifest['candidate_partition']))

In [ ]:
development = manifest.loc[manifest['candidate_partition'].eq('development')].copy()
ood_test = manifest.loc[manifest['candidate_partition'].eq('ood_test')].copy()
assert len(development) > 1 and len(ood_test) > 0
assert set(development['sample_id']).isdisjoint(set(ood_test['sample_id']))
if 'country_code' in manifest:
    dev_countries = set(development['country_code'].dropna())
    ood_countries = set(ood_test['country_code'].dropna())
    overlapping = dev_countries & ood_countries
    print('Shared candidate country codes:', sorted(overlapping))
    assert not overlapping, 'Country overlap in development and geographic test.'

# Same random stratified development split as the team's two baseline scripts.
dev_train, dev_val = train_test_split(
    development, test_size=0.2, random_state=SEED, stratify=development['class']
)
print(f'Development train: {len(dev_train):,} | Validation: {len(dev_val):,} | OOD: {len(ood_test):,}')

## 3. Naive benchmark — majority-class classifier

`DummyClassifier(strategy="most_frequent")` always predicts the most common class in the development training set. It never examines image pixels. This is the lower benchmark that a useful image classifier should beat.

In [ ]:
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(np.zeros((len(dev_train), 1)), dev_train['class'])
majority_class = dummy.classes_[dummy.class_prior_.argmax()]
print('Predicted majority class:', majority_class)

rows = []
for part_name, subset in [('development_validation', dev_val), ('ood_test', ood_test)]:
    predictions = dummy.predict(np.zeros((len(subset), 1)))
    rows.append({
        'model': 'Majority-class baseline',
        'partition': part_name,
        'n_samples': len(subset),
        'accuracy': accuracy_score(subset['class'], predictions),
        'macro_f1': f1_score(subset['class'], predictions, average='macro', zero_division=0),
    })
dummy_results = pd.DataFrame(rows)
display(dummy_results.round(4))

## 4. RGB image preprocessing and dataset checks

Images are read from the EuroSAT RGB class folders, resized to **16 × 16**, converted to RGB, flattened into 768 features, and scaled to `[0, 1]`. These settings match the team's RGB baseline script.

**Required structure:** `data/EuroSAT_RGB/EuroSAT_RGB/<class>/<sample_id>.jpg` (the cell also detects the one-level alternative `data/EuroSAT_RGB/<class>/...`). Every selected manifest sample **must** have an image: missing files now cause a clear error rather than silently removing data.

In [ ]:
candidates = [ROOT / 'data' / 'EuroSAT_RGB' / 'EuroSAT_RGB', ROOT / 'data' / 'EuroSAT_RGB']
RGB_ROOT = next((p for p in candidates if p.is_dir() and any(c.is_dir() for c in p.iterdir())), None)
if RGB_ROOT is None:
    raise FileNotFoundError(
        'EuroSAT RGB images missing. Extract the RGB dataset under '
        'data/EuroSAT_RGB/EuroSAT_RGB/<class>/ and rerun this cell.'
    )

image_paths = {}
for folder in sorted(RGB_ROOT.iterdir()):
    if folder.is_dir():
        for img in folder.iterdir():
            if img.is_file() and img.suffix.lower() in {'.jpg', '.jpeg', '.png'}:
                if img.stem in image_paths:
                    raise ValueError(f'Duplicate image sample ID: {img.stem}')
                image_paths[img.stem] = img

selected = manifest.loc[manifest['candidate_partition'].isin(['development', 'ood_test'])].copy()
missing_ids = sorted(set(selected['sample_id']) - set(image_paths))
print('RGB root:', RGB_ROOT)
print('Images indexed:', len(image_paths))
print('Images needed:', len(selected), '| Missing:', len(missing_ids))
if missing_ids:
    raise FileNotFoundError(f'Missing {len(missing_ids)} required RGB images; examples: {missing_ids[:8]}')
assert selected['sample_id'].is_unique

In [ ]:
features = []
for sample_id in selected['sample_id']:
    with Image.open(image_paths[sample_id]) as image:
        image = image.convert('RGB').resize(IMAGE_SIZE)
        features.append(np.asarray(image, dtype=np.float32).flatten() / 255.0)
X = np.asarray(features, dtype=np.float32)
y = selected['class'].to_numpy()
partitions = selected['candidate_partition'].to_numpy()
print('Features shape:', X.shape, '| Feature range:', (float(X.min()), float(X.max())))
assert X.shape == (len(selected), 16 * 16 * 3)

## 5. PCA + Random Forest — course-level baseline

PCA reduces 768 RGB pixel features to **100 components**; a **200-tree Random Forest** performs classification with balanced class weights. PCA is fit only on development training data because it is inside a scikit-learn `Pipeline`. The random seed is 42.

In [ ]:
X_dev, y_dev = X[partitions == 'development'], y[partitions == 'development']
X_ood, y_ood = X[partitions == 'ood_test'], y[partitions == 'ood_test']
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.2, random_state=SEED, stratify=y_dev
)
assert len(X_train) == len(dev_train) and len(X_val) == len(dev_val)

model = Pipeline([
    ('pca', PCA(n_components=PCA_COMPONENTS, random_state=SEED)),
    ('random_forest', RandomForestClassifier(
        n_estimators=N_TREES,
        random_state=SEED,
        n_jobs=-1,
        class_weight='balanced',
    )),
])
print(f'Training on {len(y_train):,} development images...')
model.fit(X_train, y_train)
print('Training complete.')

## 6. Evaluate the RGB baseline

For each partition, report sample count, overall accuracy, macro-F1 and the class-level precision, recall, F1, and support. OOD labels are used **only** for final evaluation, not fitting or model selection.

In [ ]:
results = []
class_results = []
all_classes = sorted(np.unique(y))
for part_name, X_eval, y_eval in [
    ('development_validation', X_val, y_val),
    ('ood_test', X_ood, y_ood),
]:
    predicted = model.predict(X_eval)
    results.append({
        'model': 'PCA + Random Forest (RGB)',
        'partition': part_name,
        'n_samples': len(y_eval),
        'accuracy': accuracy_score(y_eval, predicted),
        'macro_f1': f1_score(y_eval, predicted, average='macro', zero_division=0),
    })
    report = classification_report(y_eval, predicted, labels=all_classes,
                                   output_dict=True, zero_division=0)
    for class_name in all_classes:
        metrics = report[class_name]
        class_results.append({
            'model': 'PCA + Random Forest (RGB)',
            'partition': part_name,
            'class': class_name,
            'support': metrics['support'],
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1-score'],
        })
rf_results = pd.DataFrame(results)
per_class_results = pd.DataFrame(class_results)
display(rf_results.round(4))
display(per_class_results.round(3))

## 7. Baseline comparison and geographic performance gap

Macro-F1 averages the per-class F1 values equally. The geographic performance gap is defined here as **validation Macro-F1 − OOD Macro-F1**; a positive gap means performance declined on the candidate OOD set.

In [ ]:
combined = pd.concat([dummy_results, rf_results], ignore_index=True)
summary = combined.pivot(index='model', columns='partition', values='macro_f1')
summary['f1_drop'] = summary['development_validation'] - summary['ood_test']
print('Overall reproduced baseline comparison:')
display(summary.round(4))

per_class_pivot = per_class_results.pivot(index='class', columns='partition', values='f1')
per_class_pivot['f1_drop'] = (
    per_class_pivot['development_validation'] - per_class_pivot['ood_test']
)
per_class_pivot = per_class_pivot.sort_values('f1_drop', ascending=False)
print('Per-class gap (RGB model):')
display(per_class_pivot.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
summary[['development_validation', 'ood_test']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Macro-F1 by baseline and partition')
axes[0].set_ylabel('Macro-F1')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(['Development validation', 'Geographic OOD'])
per_class_pivot['f1_drop'].plot(kind='bar', ax=axes[1])
axes[1].set_title('RGB model: per-class Macro-F1 difference')
axes[1].set_ylabel('Validation F1 − OOD F1')
axes[1].tick_params(axis='x', rotation=65)
plt.tight_layout()
plt.show()

## 8. Compare against the team's previously stored results (optional)

The team previously stored the following results on `milestone3-backup`. They are **reference values, not freshly executed results**. The next cell checks numeric differences if the notebook has been run. Small differences may result from environment/version changes. Larger differences require investigating data paths, image matching, preprocessing, split and dependencies.

In [ ]:
reference = pd.DataFrame([
    ['Majority-class baseline', 'development_validation', 3516, 0.1359499431171786, 0.0239359038557836],
    ['Majority-class baseline', 'ood_test', 6231, 0.0913176055207831, 0.016735294117647],
    ['PCA + Random Forest (RGB)', 'development_validation', 3516, 0.7084755403868032, 0.6910487683196636],
    ['PCA + Random Forest (RGB)', 'ood_test', 6231, 0.4485636334456748, 0.4364510465773787],
], columns=['model', 'partition', 'n_samples', 'accuracy', 'macro_f1'])
comparison = combined.merge(reference, on=['model', 'partition'], suffixes=('_run', '_reference'), validate='one_to_one')
comparison['macro_f1_difference'] = comparison['macro_f1_run'] - comparison['macro_f1_reference']
comparison['accuracy_difference'] = comparison['accuracy_run'] - comparison['accuracy_reference']
comparison['sample_count_difference'] = comparison['n_samples_run'] - comparison['n_samples_reference']
display(comparison.round(6))

## 9. Export reproducible results

Save the results under new notebook-specific filenames so earlier team results are not silently overwritten. Only run this cell after successful model evaluation. Add these generated files to the final repository as appropriate.

In [ ]:
export_dir = OUTPUTS / 'milestone3_notebook'
export_dir.mkdir(parents=True, exist_ok=True)
combined.to_csv(export_dir / 'combined_baseline_results.csv', index=False)
per_class_results.to_csv(export_dir / 'rgb_per_class_results.csv', index=False)
per_class_pivot.reset_index().to_csv(export_dir / 'rgb_f1_gap_analysis.csv', index=False)
comparison.to_csv(export_dir / 'reference_comparison.csv', index=False)
print('Saved outputs:', export_dir)

## 10. Discrepancy analysis and limitations

- **Observed effect to assess:** whether the RGB baseline's macro-F1 is lower on the candidate OOD countries than on random development validation.
- **Dummy benchmark:** majority-class predictions quantify how little a classifier can achieve without image features. OOD class prevalence can change its score.
- **Class-level differences:** use the per-class comparison above to identify categories with the largest decline. In the previously saved team outputs, Pasture and Herbaceous Vegetation declined most; rerun the notebook before claiming the same result for the new execution.
- **Geographic split limitation:** development validation is a random stratified split *within* development countries, not a spatially independent holdout. The development and candidate OOD sets have disjoint country codes but patch-level spatial autocorrelation has **not** been fully ruled out.
- **Manifest limitations:** excluded and unresolved samples (including border/coast review cases) are not evaluated. Report the counts and class supports rather than treating the test set as representative of all European locations.
- **Input scope:** this course-level model uses the EuroSAT **RGB images**. Do not call this model a multispectral model.
- **Reproducibility:** preserve dependency versions, seed, image count checks, data paths and notebook outputs in the final repository or memo.

### Start / Stop / Continue (team's existing plan)
**Start:** auditing spatial overlap and finalizing geographic validation.

**Stop:** treating random validation or provisional geographic assignments as proof of geographic generalization.

**Continue:** fixed seeds, training-only PCA fitting, macro-F1, and per-class reporting.

### Lab demo walkthrough
1. Explain the land-cover classification task and geographic shift hypothesis.
2. Show the manifest partition counts and geographic holdout logic.
3. Show preprocessing and both baseline definitions.
4. Run the notebook (or show saved outputs from a verified run) and explain the results table and gap plot.
5. Discuss limitations and next steps without overclaiming geographic independence.

**Final submission:** include the notebook, necessary scripts/data-access instructions, reproduced CSVs, finalized ≤2-page memo, and GitHub release/tag. Do not claim execution is verified until every cell has run successfully with the real dataset.